# Chapter 19 - Training and Deploying TensorFlow Models at Scale

## 1. SavedModel dan TensorFlow Serving

- Setelah pelatihan, model `tf.keras` biasanya diekspor ke format **SavedModel**, yang menyimpan:
  - graph komputasi,
  - bobot model,
  - aset tambahan (mis. vocabulary atau contoh data).
- SavedModel berbentuk folder dengan struktur utama:
  - `saved_model.pb`
  - `variables/` (file bobot)
  - `assets/` (opsional).
- Model dapat dimuat kembali menggunakan:
  - `tf.saved_model.load()` atau
  - `keras.models.load_model()`.
- **TensorFlow Serving** adalah server C++ berperforma tinggi untuk deployment model, dengan fitur:
  - serving multi-versi model,
  - auto-reload dan rollback versi,
  - batching request otomatis.
- Model dapat diakses melalui **REST (HTTP JSON)** atau **gRPC**, dan mudah dijalankan menggunakan Docker.


## **Example: Export SavedModel dan Query via REST & gRPC**

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
import numpy as np

# 1) Train simple Keras model (contoh MNIST)
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])
history = model.fit(X_train, y_train, epochs=5, validation_split=0.1)

# 2) Export to SavedModel (versi 0001)
model_version = "0001"
model_name = "my_mnist_model"
model_path = os.path.join(model_name, model_version)
tf.saved_model.save(model, model_path)
# atau: model.save(model_path)  # juga SavedModel jika tanpa ekstensi .h5


In [ ]:
# 3) Query TF Serving via REST
import json
import requests
import numpy as np

X_new = X_test[:3]  # misal 3 gambar uji
input_data_json = json.dumps({
    "signature_name": "serving_default",
    "instances": X_new.tolist(),
})

SERVER_URL = "http://localhost:8501/v1/models/my_mnist_model:predict"
response = requests.post(SERVER_URL, data=input_data_json)
response.raise_for_status()
response = response.json()
y_proba = np.array(response["predictions"])
print(y_proba.round(2))


In [ ]:
# 4) Query TF Serving via gRPC
import grpc
import tensorflow as tf
from tensorflow_serving.apis.predict_pb2 import PredictRequest
from tensorflow_serving.apis import prediction_service_pb2_grpc

channel = grpc.insecure_channel("localhost:8500")
predict_service = prediction_service_pb2_grpc.PredictionServiceStub(channel)

request = PredictRequest()
request.model_spec.name = model_name
request.model_spec.signature_name = "serving_default"
input_name = model.input_names[0]
request.inputs[input_name].CopyFrom(tf.make_tensor_proto(X_new))

response = predict_service.Predict(request, timeout=10.0)
output_name = model.output_names[0]
outputs_proto = response.outputs[output_name]
y_proba = tf.make_ndarray(outputs_proto)
print(y_proba.round(2))


## 2. Google Cloud AI Platform: Prediction Service & Hyperparameter Tuning

- **Google Cloud AI Platform (Vertex AI)** menyediakan layanan hosting prediction berbasis TensorFlow Serving dengan:
  - autoscaling,
  - logging,
  - integrasi langsung dengan Google Cloud Storage (GCS).
- Alur deployment umum:
  1. Upload SavedModel ke GCS.
  2. Buat *Model* di AI Platform.
  3. Buat *Version* yang menunjuk ke path SavedModel  
     (mis. `gs://bucket/model_name/0001/`).
- Akses prediction dilakukan melalui **REST API** menggunakan Google API Client Library.
- Autentikasi menggunakan **service account** dengan key JSON yang diset melalui
  `GOOGLE_APPLICATION_CREDENTIALS`.
- AI Platform mendukung **hyperparameter tuning** berbasis Bayesian optimization
  (Google Vizier), dikonfigurasi lewat file YAML dan menggunakan metrik dari TensorBoard
  (misalnya accuracy atau loss).

## **Example: Client Prediction ke AI Platform & Hyperparameter Tuning**

In [ ]:
import os
import numpy as np
import googleapiclient.discovery

# 1) Autentikasi service account
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "my_service_account_key.json"

# 2) Bangun resource untuk AI Platform
project_id = "your-gcp-project-id"
model_id = "my_mnist_model"
model_path = f"projects/{project_id}/models/{model_id}"

ml_resource = googleapiclient.discovery.build("ml", "v1").projects()

output_name = "dense_1"  # sesuaikan dengan nama output layer SavedModel

def predict(X):
    body = {
        "signature_name": "serving_default",
        "instances": X.tolist(),
    }
    request = ml_resource.predict(name=model_path, body=body)
    response = request.execute()
    if "error" in response:
        raise RuntimeError(response["error"])
    return np.array([pred[output_name] for pred in response["predictions"]])

Y_probas = predict(X_new)
print(np.round(Y_probas, 2))


In [ ]:
# Di training script: gunakan argumen CLI & TensorBoard callback
import argparse
import tensorflow as tf
from tensorflow import keras

parser = argparse.ArgumentParser()
parser.add_argument("--n_layers", type=int, default=20)
parser.add_argument("--momentum", type=float, default=0.9)
parser.add_argument("--job-dir", type=str, default="/tmp/jobdir")
args, _ = parser.parse_known_args()

model = keras.models.Sequential(
    [keras.layers.Flatten(input_shape=[28, 28])] +
    [keras.layers.Dense(100, activation="relu") for _ in range(args.n_layers)] +
    [keras.layers.Dense(10, activation="softmax")]
)
optimizer = keras.optimizers.SGD(momentum=args.momentum)
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=optimizer,
              metrics=["accuracy"])

tb_cb = keras.callbacks.TensorBoard(
    log_dir=args.job_dir,
    update_freq="batch"
)

model.fit(X_train, y_train, epochs=10, validation_split=0.1,
          callbacks=[tb_cb])


## 3. TFLite dan TF.js: Deployment ke Mobile, Embedded, dan Browser

- **TensorFlow Lite (TFLite)** ditujukan untuk deployment model di mobile dan embedded dengan fokus pada:
  - ukuran model lebih kecil,
  - latency dan konsumsi energi rendah,
  - kompatibilitas dengan keterbatasan hardware (RAM, CPU).
- Model `SavedModel` atau `tf.keras` dikonversi menjadi file `.tflite` berbasis **FlatBuffers** menggunakan TFLite Converter.
- Proses konversi mencakup berbagai optimisasi graph, seperti:
  - penghapusan operasi training-only,
  - fusion Batch Normalization,
  - penyederhanaan graph.
- **Post-training quantization** (mis. float32 → int8) sangat mengurangi ukuran model dan mempercepat inference.
- Skema kuantisasi simetris memetakan rentang float `[-m, m]` ke integer `[-127, 127]`, memungkinkan *full integer inference* dengan latency sangat rendah.
- **Quantization-aware training (QAT)** menambahkan operasi kuantisasi palsu saat training sehingga akurasi model lebih terjaga setelah dikonversi.
- Untuk aplikasi web, **TensorFlow.js** menjalankan model langsung di browser dari format `model.json` dan shard bobot biner.
- TF.js cocok untuk kebutuhan latency rendah, penggunaan offline, dan perlindungan privasi data (inference di sisi klien).


## **Example: Convert ke TFLite dengan Quantization dan Run Interpreter**

In [ ]:
import tensorflow as tf

# Asumsikan model Keras sudah dilatih
saved_model_path = "my_mnist_model/0001"

# 1) Konversi SavedModel ke TFLite basic
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
tflite_model = converter.convert()
with open("model_basic.tflite", "wb") as f:
    f.write(tflite_model)

# 2) Post-training quantization (OPTIMIZE_FOR_SIZE)
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]
tflite_quant_model = converter.convert()
with open("model_quant.tflite", "wb") as f:
    f.write(tflite_quant_model)

# 3) Jalankan TFLite model di Python dengan TFLite Interpreter
import numpy as np

interpreter = tf.lite.Interpreter(model_path="model_quant.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

X_sample = X_test[:1].astype(np.float32)
interpreter.set_tensor(input_details[0]["index"], X_sample)
interpreter.invoke()
output_data = interpreter.get_tensor(output_details[0]["index"])
print(output_data.round(2))


## 4. GPU/TPU, Manajemen Memori, dan Multi-Device Placement

- Training deep model di CPU lambat; **GPU/TPU** mempercepat operasi intensif seperti convolution dan matrix multiplication.
- Untuk GPU lokal diperlukan:
  - driver NVIDIA,
  - CUDA,
  - cuDNN.
- Ketersediaan GPU dapat dicek dengan:
  - `tf.test.is_gpu_available()`
  - `tf.config.experimental.list_physical_devices('GPU')`.
- Alternatif praktis:
  - Google Colab (GPU/TPU),
  - cloud VM GPU (mis. GCP Deep Learning VM).
- Secara default, TensorFlow mengalokasikan seluruh memori GPU pertama.
- Opsi `set_virtual_device_configuration` dan `set_memory_growth` memungkinkan:
  - pembatasan memori GPU,
  - alokasi memori bertahap (*on-demand*),
  - berbagi GPU antar proses.
- Penempatan operasi dan variabel ke device dilakukan otomatis, tetapi bisa dioverride dengan:
  - `tf.device("/gpu:1")`
  - `tf.device("/cpu:0")`.


## **Example: Cek GPU, Atur Memory, dan Split Virtual GPU**

In [ ]:
import tensorflow as tf

# 1) Cek GPU
print(tf.config.experimental.list_physical_devices("GPU"))

# 2) Set memory growth supaya TF tidak grab semua RAM GPU
gpus = tf.config.experimental.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# 3) Batasi RAM GPU (misal 2 GiB per virtual device)
for gpu in gpus:
    tf.config.experimental.set_virtual_device_configuration(
        gpu,
        [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=2048)]
    )

# 4) Atau split 1 GPU menjadi 2 virtual GPU (2 GiB + 2 GiB)
physical_gpus = tf.config.experimental.list_physical_devices("GPU")
if physical_gpus:
    tf.config.experimental.set_virtual_device_configuration(
        physical_gpus[0],
        [
            tf.config.experimental.VirtualDeviceConfiguration(memory_limit=2048),
            tf.config.experimental.VirtualDeviceConfiguration(memory_limit=2048),
        ],
    )
    logical_gpus = tf.config.experimental.list_logical_devices("GPU")
    print("Logical GPUs:", logical_gpus)


In [ ]:
# 5) Contoh penempatan variabel di CPU vs GPU
a = tf.Variable(42.0)  # float → GPU default bila ada
b = tf.Variable(42)    # int → CPU
print("a on:", a.device)
print("b on:", b.device)

with tf.device("/cpu:0"):
    c = tf.Variable(3.14)
print("c on:", c.device)


## 5. Distribution Strategies: Multi-GPU & Multi-Server Training

- Skala training diperbesar dengan **data parallelism**:
  - setiap device memiliki replika model identik,
  - masing-masing memproses mini-batch berbeda,
  - gradient di-aggregate (mis. **AllReduce**) sebelum update parameter.
- **MirroredStrategy**:
  - untuk multi-GPU dalam satu mesin,
  - menggunakan AllReduce (NCCL) untuk merata-ratakan gradient,
  - sederhana dan efisien untuk single-node multi-GPU.
- **CentralStorageStrategy**:
  - parameter model disimpan di satu device (biasanya CPU),
  - GPU hanya menghitung gradient,
  - cocok untuk model kecil atau keterbatasan memori GPU.
- Untuk **multi-server (cluster)**:
  - TensorFlow menyediakan **MultiWorkerMirroredStrategy** dan **ParameterServerStrategy**.
  - Konfigurasi cluster didefinisikan melalui variabel environment `TF_CONFIG`.
  - Peran umum: `chief`, `worker`, `parameter_server`, dan `evaluator`.
- **MultiWorkerMirroredStrategy**:
  - memperluas MirroredStrategy ke banyak mesin,
  - menggunakan AllReduce lintas server,
  - cocok untuk training sinkron berskala besar.
- **ParameterServerStrategy**:
  - worker menghitung gradient,
  - parameter server menyimpan dan memperbarui bobot,
  - lebih fleksibel untuk skenario asinkron dan cluster besar.
- Job distributed dapat dijalankan di cloud (mis. AI Platform / Vertex AI) dengan banyak worker dan PS.
- Model yang dilatih secara distributed tetap disimpan sebagai **SavedModel standar**:
  - dapat diload dan digunakan tanpa konteks distribusi,
  - atau dijalankan kembali dengan strategi distribusi untuk inference paralel multi-GPU.


## **Example: Multi‑GPU Training dengan MirroredStrategy**

In [ ]:
import tensorflow as tf
from tensorflow import keras

# 1) Buat strategy di semua GPU lokal
strategy = tf.distribute.MirroredStrategy()  # atau MirroredStrategy(["/gpu:0","/gpu:1"])
print("Num replicas in sync:", strategy.num_replicas_in_sync)

# 2) Bangun & compile model di dalam scope strategy
with strategy.scope():
    model = keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.Dense(300, activation="relu"),
        keras.layers.Dense(100, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer=keras.optimizers.Adam(),
                  metrics=["accuracy"])

# Batch size harus kelipatan jumlah replika
global_batch_size = 256
history = model.fit(X_train, y_train,
                    epochs=10,
                    batch_size=global_batch_size,
                    validation_split=0.1)

# 3) Simpan model seperti biasa
model.save("mnist_multi_gpu.h5")

# 4) Load untuk inference saja (single device)
single_model = keras.models.load_model("mnist_multi_gpu.h5")
y_proba_single = single_model.predict(X_test[:512])

# 5) Atau load lagi dalam scope untuk inference multi-GPU
with strategy.scope():
    dist_model = keras.models.load_model("mnist_multi_gpu.h5")
y_proba_dist = dist_model.predict(X_test[:global_batch_size])


In [ ]:
# Example skeleton: CentralStorageStrategy
strategy = tf.distribute.experimental.CentralStorageStrategy()

with strategy.scope():
    model = keras.models.Sequential([
        keras.layers.Flatten(input_shape=[28, 28]),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(loss="sparse_categorical_crossentropy",
                  optimizer="adam",
                  metrics=["accuracy"])

model.fit(X_train, y_train,
          epochs=5,
          batch_size=global_batch_size,
          validation_split=0.1)
